## Cài môi trường & kiểm tra key + prompt gốc

Thực hiện cài `vllm`, `gcc` và xem chính xác tên cột trong `humaneval_normsrcpseudo.json` cũng như tên `model / prompt` trong `sk2decompile_inf.py`

In [1]:
!pip install -q vllm pandas tqdm editdistance
!apt-get update -qq && apt-get install -y -qq gcc

import json

EVAL_DIR = "/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/LLM4Decompile/sk2decompile/evaluation"
HE_PATH = f"{EVAL_DIR}/data/humaneval_normsrcpseudo.json"

with open(HE_PATH, "r", encoding="utf-8") as f:
    he_data = json.load(f)

c_data = [d for d in he_data if d.get("language") == "c"]
print(f"Tổng mẫu: {len(he_data)} | Mẫu C: {len(c_data)}")
print("Các mức opt:", sorted(set(d.get("opt") for d in c_data)))
print("Các key:", list(c_data[0].keys()))
print("Có unit test ở mẫu 0:", len(str(c_data[0].get("test", ""))) > 0)

print("\n=== PROMPT & MODEL TRONG sk2decompile_inf.py ===")
!grep -n -C 4 -E "prompt|template|model|ident|struct" {EVAL_DIR}/sk2decompile_inf.py | head -n 45

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is i

In [2]:
!sed -n '25,130p' /kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/LLM4Decompile/sk2decompile/evaluation/sk2decompile_inf.py

    arg_parser.add_argument("--output_path", type=str, default='./result/sk2decompile')
    arg_parser.add_argument("--only_save", type=int, default=0)
    arg_parser.add_argument("--strip", type=int, default=1)
    arg_parser.add_argument("--language", type=str, default='c')
    args = arg_parser.parse_args()

    before = "# This is the assembly code:\n"
    after = "\n# What is the source code?\n"

    if args.dataset_path.endswith('.json'):
        with open(args.dataset_path, "r") as f:
            print("===========")
            print(f"Loading dataset from {args.dataset_path}")
            print("===========")
            samples = json.load(f)
    elif args.dataset_path.endswith('.jsonl'):
        samples = []
        with open(args.dataset_path, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    samples.append(json.loads(line))


    tokenizer = AutoTokenizer.from_pretrained(args.model_path)
    if args.stop_

## Phase 1 thinking

In [3]:
%%writefile /kaggle/working/run_phase1.py
import json
from vllm import LLM, SamplingParams

HE_PATH = "/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/LLM4Decompile/sk2decompile/evaluation/data/humaneval_normsrcpseudo.json"
OUT_P1 = "/kaggle/working/he_phase1.jsonl"
MODEL_P1 = "LLM4Binary/sk2decompile-struct-6.7b"

MAX_MODEL_LEN = 8192
MAX_NEW_TOKENS = 1536
MAX_PROMPT_TOKENS = MAX_MODEL_LEN - MAX_NEW_TOKENS

with open(HE_PATH, "r", encoding="utf-8") as f:
    samples = [d for d in json.load(f) if d.get("language") == "c"]

llm = LLM(
    model=MODEL_P1,
    tensor_parallel_size=2,
    dtype="half",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=0.85,
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()

before = "# This is the assembly code:\n"
after = "\n# What is the source code?\n"
prompts = []
truncated_cnt = 0
for s in samples:
    code_ids = tokenizer.encode(s["ida_pseudo_norm"].strip(), add_special_tokens=False)
    if len(code_ids) > MAX_PROMPT_TOKENS - 64:
        code_ids = code_ids[: MAX_PROMPT_TOKENS - 64]
        truncated_cnt += 1
    code_str = tokenizer.decode(code_ids, skip_special_tokens=True)
    prompts.append(before + code_str + after)

print(f"Đã chuẩn bị {len(prompts)} prompts (số mẫu bị cắt bớt do quá dài: {truncated_cnt})")

params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
    stop=[tokenizer.eos_token] if tokenizer.eos_token else None,
)

outputs = llm.generate(prompts, params)

with open(OUT_P1, "w", encoding="utf-8") as f_out:
    for s, out in zip(samples, outputs):
        raw_p1 = out.outputs[0].text
        clean_p1 = raw_p1.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t").strip()
        s["phase1_ir"] = clean_p1
        f_out.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"Hoàn tất Pha 1: {len(samples)} mẫu -> {OUT_P1}")

Writing /kaggle/working/run_phase1.py


In [4]:
!python /kaggle/working/run_phase1.py

W1122 02:16:10.219000 492 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1122 02:16:10.220000 492 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1122 02:16:10.220000 492 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=492) INFO 11-22 02:16:39 [startup.py:96] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'LLM4Binary/sk2decompile-struct-6.7b'}
config.json: 100%|█████████████████████████████| 778/778 [00:00<00:00, 2.51MB/s]
(MainProcess pid=492) INFO 11-22 02:16:57 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=492) INFO 11-22 02:16:57 [model.py:2557] Downcasting torch.float32 to torch.float16.
(MainProcess pid=492) INFO 11-22 02:16:57 [model.py:2214] Using max mo

## Phase 2 thinking

In [5]:
%%writefile /kaggle/working/run_phase2.py
import json
from vllm import LLM, SamplingParams

IN_P1 = "/kaggle/working/he_phase1.jsonl"
OUT_P2 = "/kaggle/working/sk2_humaneval_recovered.jsonl"
MODEL_P2 = "LLM4Binary/sk2decompile-ident-6.7b"

MAX_MODEL_LEN = 8192
MAX_NEW_TOKENS = 1536
MAX_PROMPT_TOKENS = MAX_MODEL_LEN - MAX_NEW_TOKENS

with open(IN_P1, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f if line.strip()]

llm = LLM(
    model=MODEL_P2,
    tensor_parallel_size=2,
    dtype="half",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=0.85,
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()

before_rec = "# This is the normalized code:\n"
after_rec = "\n# What is the source code?\n"
prompts = []
truncated_cnt = 0
for s in samples:
    code_ids = tokenizer.encode(s["phase1_ir"].strip(), add_special_tokens=False)
    if len(code_ids) > MAX_PROMPT_TOKENS - 64:
        code_ids = code_ids[: MAX_PROMPT_TOKENS - 64]
        truncated_cnt += 1
    code_str = tokenizer.decode(code_ids, skip_special_tokens=True)
    prompts.append(before_rec + code_str + after_rec)

print(f"Đã chuẩn bị {len(prompts)} prompts (số mẫu bị cắt bớt: {truncated_cnt})")

params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
    stop=[tokenizer.eos_token] if tokenizer.eos_token else None,
)

outputs = llm.generate(prompts, params)

with open(OUT_P2, "w", encoding="utf-8") as f_out:
    for s, out in zip(samples, outputs):
        raw_p2 = out.outputs[0].text
        clean_p2 = raw_p2.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t").strip()
        s["recovered_code"] = clean_p2
        f_out.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"Hoàn tất Pha 2: {len(samples)} mẫu -> {OUT_P2}")

Writing /kaggle/working/run_phase2.py


In [6]:
!python /kaggle/working/run_phase2.py

W1122 02:38:03.333000 1297 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1122 02:38:03.333000 1297 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1122 02:38:03.334000 1297 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=1297) INFO 11-22 02:38:18 [startup.py:96] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'LLM4Binary/sk2decompile-ident-6.7b'}
config.json: 100%|█████████████████████████████| 754/754 [00:00<00:00, 2.91MB/s]
(MainProcess pid=1297) INFO 11-22 02:38:19 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=1297) INFO 11-22 02:38:19 [model.py:2557] Downcasting torch.float32 to torch.float16.
(MainProcess pid=1297) INFO 11-22 02:38:19 [model.py:2214] Using 

## Đánh giá `Re-compilation`, `Re-executability`, `Edit Similarity` & xuất CSV

In [7]:
import os
import re
import json
import tempfile
import subprocess
import pandas as pd
import editdistance
from multiprocessing import Pool
from tqdm import tqdm

INPUT_JSONL = "/kaggle/working/sk2_humaneval_recovered.jsonl"
CSV_OUT = "/kaggle/working/sk2_humaneval_metrics.csv"

# Header chuẩn bổ sung mock định nghĩa để tăng khả năng tương thích biên dịch
STD_HEADERS = """
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <stdbool.h>
#include <stddef.h>
#include <string.h>
#include <math.h>
#include <limits.h>
#include <ctype.h>
#include <assert.h>
"""

# Danh sách từ khóa nguyên thủy trong C cần tách nếu bị dính chữ
C_KEYWORDS = [
    "long long", "const char", "unsigned int", "unsigned long", "unsigned char",
    "char", "int", "float", "double", "void", "short", "long", "size_t", "bool"
]

def sanitize_c_code(code: str, target_func_name: str) -> str:
    """
    Chuẩn hóa code C bị dính token từ đầu ra Pha 2 của mô hình:
    - Xóa tiền tố C++ ::
    - Tách các từ khóa dính liền (intlen, constchar, inti, return1, elseif,...)
    - Khôi phục chính xác tên hàm thành target_func_name
    """
    if not code:
        return ""
    
    # 1. Giải mã ký tự BPE đặc biệt nếu còn sót
    text = code.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t").strip()
    
    # 2. Bóc code block nếu có
    m = re.search(r"```(?:c|cpp)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    text = m.group(1).strip() if m else text.strip()
    
    # 3. Xóa các dấu scope C++ (::)
    text = re.sub(r"::\s*", "", text)
    
    # 4. Sửa các lỗi từ khóa điều khiển dính liền phổ biến
    text = re.sub(r"\breturn0\b", "return 0", text)
    text = re.sub(r"\breturn1\b", "return 1", text)
    text = re.sub(r"\breturn([A-Za-z0-9_]+);", r"return \1;", text)
    text = re.sub(r"\belseif\b", "else if", text)
    
    # 5. Tách kiểu dữ liệu đặc biệt bị dính
    text = re.sub(r"\bintlonglong\b", "long long ", text)
    text = re.sub(r"\blonglong\b", "long long ", text)
    text = re.sub(r"\bconstchar\b", "const char ", text)
    
    # Tách kiểu dữ liệu dính liền với tên biến: vd 'intlen' -> 'int len', 'inti' -> 'int i', 'floateps' -> 'float eps'
    type_pattern = r"\b(int|float|double|char|void|short|bool)(?=[A-Za-z_])"
    text = re.sub(type_pattern, r"\1 ", text)
    
    # 6. Khôi phục tên hàm chính xác (thay hàm được sinh ra đầu tiên bằng target_func_name)
    if "(" in text and target_func_name:
        # Tìm vị trí dấu '(' đầu tiên đại diện cho khai báo hàm
        idx_paren = text.find("(")
        header = text[:idx_paren]
        rest = text[idx_paren:]
        
        # Tìm identifier cuối cùng trước dấu '('
        match_id = list(re.finditer(r"[A-Za-z_]\w*", header))
        if match_id:
            last_match = match_id[-1]
            old_name = last_match.group(0)
            # Nếu tên nhận dạng khác kiểu dữ liệu thì thay bằng target_func_name
            if old_name not in ["int", "void", "char", "float", "double", "long", "short"]:
                start, end = last_match.span()
                header = header[:start] + target_func_name + header[end:]
                text = header + rest
                
    return text

def calc_edit_sim(pred, ref):
    if not pred and not ref:
        return 100.0
    max_len = max(len(pred), len(ref), 1)
    return (1.0 - editdistance.eval(pred, ref) / max_len) * 100.0

def eval_one(r):
    target_name = r.get("func_name", "")
    raw_code = r.get("recovered_code", "")
    code = sanitize_c_code(raw_code, target_name)
    
    ref = r.get("func", "") or ""
    func_dep = str(r.get("func_dep", "") or "").strip()
    func_test = str(r.get("test", "") or "").strip()

    comp_ok, exe_ok = False, False
    if code:
        full_src = f"{STD_HEADERS}\n{func_dep}\n{code}\n{func_test}\n"
        with tempfile.TemporaryDirectory() as tmp:
            c_file = os.path.join(tmp, "prog.c")
            bin_file = os.path.join(tmp, "prog")
            with open(c_file, "w", encoding="utf-8") as f:
                f.write(full_src)
            try:
                # Biên dịch gcc với các flag tương thích chuẩn C99 và nới lỏng ép kiểu con trỏ
                subprocess.run(
                    ["gcc", "-O0", "-std=gnu99", "-Wno-incompatible-pointer-types", "-Wno-int-conversion", c_file, "-o", bin_file, "-lm"],
                    check=True, timeout=10, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
                )
                comp_ok = True
            except Exception:
                comp_ok = False

            if comp_ok and func_test:
                try:
                    subprocess.run([bin_file], check=True, timeout=10,
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    exe_ok = True
                except Exception:
                    exe_ok = False

    return {
        "opt": r.get("opt", "unknown").lstrip("-"),
        "re_compilation": int(comp_ok) * 100,
        "re_executability": int(exe_ok) * 100,
        "edit_similarity": calc_edit_sim(code, ref),
    }

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Đang tiến hành đánh giá chuẩn hóa trên {len(records)} mẫu...")
with Pool(processes=4) as pool:
    rows = list(tqdm(pool.imap(eval_one, records), total=len(records)))

df = pd.DataFrame(rows)
summary = df.groupby("opt").mean().reset_index()
overall = pd.DataFrame([{
    "opt": "ALL_AVG",
    "re_compilation": df["re_compilation"].mean(),
    "re_executability": df["re_executability"].mean(),
    "edit_similarity": df["edit_similarity"].mean(),
}])
summary = pd.concat([summary, overall], ignore_index=True).round(2)
summary.to_csv(CSV_OUT, index=False)

print("\n=== KẾT QUẢ SAU KHI FIX GAP TRÊN HUMANEVAL-C ===")
print(summary)

Đang tiến hành đánh giá chuẩn hóa trên 567 mẫu...


100%|██████████| 567/567 [00:31<00:00, 17.75it/s]


=== KẾT QUẢ SAU KHI FIX GAP TRÊN HUMANEVAL-C ===
       opt  re_compilation  re_executability  edit_similarity
0       O0           55.49             40.24            40.24
1       O1           55.56             29.41            33.55
2       O2           50.36             21.17            32.83
3       O3           47.79             14.16            31.94
4  ALL_AVG           52.73             27.51            34.99
